## Day 22

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Airbnb_open_data_processed.csv")

In [3]:
df.head()

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country code,...,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,has_reviews
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,US,...,966.0,193.0,10.0,9.0,2021-10-19,0.21,4.0,6.0,286.0,1
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,US,...,142.0,28.0,30.0,45.0,2022-05-21,0.38,4.0,2.0,228.0,1
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,Unknown,Elise,Manhattan,Harlem,40.80902,-73.94190,US,...,620.0,124.0,3.0,0.0,NaN,0.00,5.0,1.0,352.0,0
3,1002755,Unknown,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,US,...,368.0,74.0,30.0,270.0,2019-07-05,4.64,4.0,1.0,322.0,1
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,US,...,204.0,41.0,10.0,9.0,2018-11-19,0.10,3.0,1.0,289.0,1


In [4]:
df_sample = df.sample(frac=0.3, random_state=42)

To manage computational constraints caused by high-cardinality categorical features and dataset size, model training was performed on a representative sample with controlled tree depth and estimator count.

In [5]:
y = df_sample['price']

X = df_sample.drop(columns=[
    'price',
    'id',
    'host id',
    'NAME',
    'host name',
    'neighbourhood'
])

In [6]:
num = X.select_dtypes(include=['int64', 'float64']).columns
cat = X.select_dtypes(include='object').columns

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), cat)
])

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    ))
])

In [14]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

mae, rmse, r2

(3.445183269637685, np.float64(23.647573907625645), 0.9949364924547077)

In [ ]:
-

In [17]:
import pandas as pd

feature_names = model.named_steps['prep'].get_feature_names_out()
importances = model.named_steps['rf'].feature_importances_

fi = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

fi.head(10)

,feature,importance
3,num__service fee,0.998659
0,num__lat,0.000197
9,num__availability 365,0.000161
1,num__long,0.000149
6,num__reviews per month,0.000120
4,num__minimum nights,0.000115
8,num__calculated host listings count,0.000102
2,num__Construction year,0.000100
5,num__number of reviews,0.000081
7,num__review rate number,0.000055


In [18]:
import joblib

joblib.dump(model, "airbnb_price_model.pkl")

['airbnb_price_model.pkl']

## What we did today:

**Moved from EDA → Modeling**
- Shifted focus from understanding data to building an actual ML model
- Defined a clear business target: predicting Airbnb listing prices.
- 
**Prepared Data for ML**
  
**Built a ML Pipeline**

**Trained the Model Successfully**

**Reached the Evaluation Stage**